In [1]:
# Step 0 setup: find the project root and put src/ on the import path
import sys, importlib, inspect
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").is_dir():
    if root == root.parent:
        raise FileNotFoundError("Could not find the project's src/ folder")
    root = root.parent
sys.path.insert(0, str(root / "src"))

import config, features
importlib.reload(config)
importlib.reload(features)
from config import TRAIN_PATH, VALID_PATH, DECEMBER_PATH, DATE_COL

In [2]:
# Check A: features.py no longer defines DATE_COL itself
src = inspect.getsource(features)
print("features defines DATE_COL itself:", 'DATE_COL = "date"' in src)   # expect False
print("features.DATE_COL =", repr(features.DATE_COL))                    # expect 'date'

# Check B: every distance in all three files gets a band (raw files; cleaning never changes distance)
files = {"train": pd.read_csv(TRAIN_PATH),
         "validation": pd.read_csv(VALID_PATH),
         "december": pd.read_csv(DECEMBER_PATH)}
for name, df in files.items():
    bands = features.distance_band(df["distance"])   # raises if any band is NaN
    print(f"\n{name}: rows={len(df):,}  distance min={df['distance'].min():.1f}  max={df['distance'].max():.1f}")
    print(bands.value_counts(sort=False).to_string())

# Check C: the safety net actually fires on bad input
try:
    features.distance_band(pd.Series([100.0, 4500.0, None]))
except ValueError as e:
    print("\nSafety check works:", e)
else:
    print("\nPROBLEM: no error raised")

features defines DATE_COL itself: False
features.DATE_COL = 'date'

train: rows=48,000  distance min=70.0  max=3439.8
distance
(0, 250]         2713
(250, 500]       7577
(500, 1000]     15055
(1000, 2000]    15108
(2000, 4000]     7547

validation: rows=12,000  distance min=70.0  max=3325.7
distance
(0, 250]         683
(250, 500]      1881
(500, 1000]     3735
(1000, 2000]    3792
(2000, 4000]    1909

december: rows=31  distance min=360.0  max=360.0
distance
(0, 250]         0
(250, 500]      31
(500, 1000]      0
(1000, 2000]     0
(2000, 4000]     0

Safety check works: 2 distance(s) missing or outside 0-4000 mi, e.g. [4500.0, nan]


In [3]:
# ===== Step 1: time splits =====
import validation
importlib.reload(validation)

raw_train = pd.read_csv(TRAIN_PATH)
splits = {}
for fold in [validation.DECISION_FOLD, validation.MAIN_FOLD]:
    tr, te = validation.time_split(raw_train, fold)
    splits[fold.name] = (tr, te)
    print(validation.split_summary(tr, te, fold).to_string(index=False), "\n")

dec_tr, dec_te = splits["decision"]
main_tr, main_te = splits["main"]

# Cross-checks
print("main covers every row (train+test = 48,000):", len(main_tr) + len(main_te) == len(raw_train))
print("Aug + Sep-Oct = 14,282 (Step 13's Aug-Oct holdout):", len(dec_te) + len(main_te))
dec_dates = pd.to_datetime(pd.concat([dec_tr, dec_te])[DATE_COL])
print("decision fold touches Sep-Oct:", (dec_dates >= "2025-09-01").any())   # expect False
print("decision test ids inside main TRAIN:", set(dec_te["load_id"]) <= set(main_tr["load_id"]))  # expect True
print("raw_train date still text (untouched):", raw_train[DATE_COL].dtype)    # expect object

# Safety check: an overlapping fold must be refused
bad = validation.Fold("bad", "2025-01-01", "2025-08-31", "2025-08-15", "2025-10-31")
try:
    validation.time_split(raw_train, bad)
    print("PROBLEM: overlapping fold was NOT refused")
except ValueError as e:
    print("overlap refused as expected:", e)

    fold  side  rows      first       last
decision train 33718 2025-01-01 2025-07-31
decision  test  4759 2025-08-01 2025-08-31 

fold  side  rows      first       last
main train 38477 2025-01-01 2025-08-31
main  test  9523 2025-09-01 2025-10-31 

main covers every row (train+test = 48,000): True
Aug + Sep-Oct = 14,282 (Step 13's Aug-Oct holdout): 14282
decision fold touches Sep-Oct: False
decision test ids inside main TRAIN: True
raw_train date still text (untouched): object
overlap refused as expected: Fold 'bad': training must end before testing starts


In [4]:
# ===== Step 2: metrics =====
importlib.reload(validation)
import numpy as np

# Toy 1: compare with your hand calculation
true1, pred1 = [100, 200, 400], [110, 180, 400]
print("Toy 1:", {k: round(v, 3) for k, v in validation.score(true1, pred1).items()})

# Toy 2: one big miss among four loads. How do MAE and RMSE react?
true2, pred2 = [100, 100, 100, 100], [100, 100, 100, 200]
print("Toy 2:", {k: round(v, 3) for k, v in validation.score(true2, pred2).items()})

# Toy 3: a label error with a falsely LOW true price. How does MAPE react?
true3, pred3 = [200, 200, 50], [200, 200, 200]
print("Toy 3:", {k: round(v, 3) for k, v in validation.score(true3, pred3).items()})

# Cross-check against scikit-learn on 1,000 random rates
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
rng = np.random.default_rng(42)
t = rng.uniform(200, 5000, 1000)
p = t * rng.uniform(0.7, 1.3, 1000)
print("\nmatches sklearn MAE: ", np.isclose(validation.mae(t, p), mean_absolute_error(t, p)))
print("matches sklearn MAPE:", np.isclose(validation.mape(t, p), mean_absolute_percentage_error(t, p) * 100))
print("matches sklearn RMSE:", np.isclose(validation.rmse(t, p), root_mean_squared_error(t, p)))

# Safety checks: bad inputs must stop the run
for label, (tt, pp) in {"zero true rate": ([0, 100], [50, 100]),
                        "NaN prediction": ([100, 100], [np.nan, 100]),
                        "length mismatch": ([100, 100], [100])}.items():
    try:
        validation.score(tt, pp)
        print(f"PROBLEM: {label} NOT caught")
    except ValueError as e:
        print(f"{label} caught: {e}")

Toy 1: {'MAE': 10.0, 'MAPE_%': 6.667, 'RMSE': 12.91, 'bias': -3.333}
Toy 2: {'MAE': 25.0, 'MAPE_%': 25.0, 'RMSE': 50.0, 'bias': 25.0}
Toy 3: {'MAE': 50.0, 'MAPE_%': 100.0, 'RMSE': 86.603, 'bias': 50.0}

matches sklearn MAE:  True
matches sklearn MAPE: True
matches sklearn RMSE: True
zero true rate caught: True rates must be > 0 (MAPE divides by them)
NaN prediction caught: NaN or infinite values in true or predicted rates
length mismatch caught: Length mismatch: (2,) true vs (1,) predicted


In [ ]:
# ===== Step 3a: full-file reproduction of the 677 =====
importlib.reload(config); importlib.reload(features)   # config first: features imports from it
import baselines; importlib.reload(baselines)

learned_full = features.fit_cleaning(raw_train)
clean_train = features.clean(raw_train, learned_full)

table_full = baselines.fit_price_table(clean_train)
sizes = baselines.rpm(clean_train).groupby(baselines._segment_keys(clean_train)).size()
sizes.index.names = table_full.index.names
print(pd.DataFrame({"median_rpm": table_full.round(3), "rows": sizes}).to_string())

flags_full = baselines.flag_label_errors(clean_train, table_full)
full_ids = set(clean_train.loc[flags_full["is_error"].to_numpy(), "load_id"])
print(f"\nlow: {flags_full['low'].sum()}  high: {flags_full['high'].sum()}  total: {flags_full['is_error'].sum()}")

In [ ]:
# ===== Step 3b: label-error flags per training fold =====
for fold in [validation.DECISION_FOLD, validation.MAIN_FOLD]:
    tr_raw, _ = validation.time_split(raw_train, fold)
    tr = features.clean(tr_raw, features.fit_cleaning(tr_raw))     # cleaning learned from the fold
    kept, flags = baselines.drop_label_errors(tr)                   # table learned from the fold
    ids = set(tr.loc[flags["is_error"].to_numpy(), "load_id"])
    print(f"{fold.name}: train {len(tr):,}  flagged {len(ids)} "
          f"(low {flags['low'].sum()}, high {flags['high'].sum()})  kept {len(kept):,}  | "
          f"full-file flags in this window: {len(full_ids & set(tr['load_id']))}  "
          f"| shared with them: {len(ids & full_ids)}")

In [ ]:
# ===== Step 3c: baseline mechanics (NOT scores: predicting the rows they learned from) =====
b0 = baselines.GlobalMedianRate().fit(clean_train)
b1 = baselines.SegmentMedianRpm().fit(clean_train)
for m in (b0, b1):
    p = m.predict(clean_train)
    print(f"{m.name}: n={len(p):,}  NaN={np.isnan(p).sum()}  "
          f"min={p.min():,.0f}  max={p.max():,.0f}  all>0={bool((p > 0).all())}")

# December sanity: B1 ignores the date, so expect ONE value for all 31 days.
# Phase 2 sanity band: 750-950.
clean_valid = features.clean(pd.read_csv(VALID_PATH), learned_full)
dec_clean = features.clean(pd.read_csv(DECEMBER_PATH), learned_full,
                           market_by_date=features.daily_market_index(clean_valid))
dec_p = b1.predict(dec_clean)
print(f"\nB1 December: unique values {np.unique(dec_p.round(2))}  "
      f"inside 750-950: {bool(((dec_p >= 750) & (dec_p <= 950)).all())}")

# Safety: a segment missing from the table must stop the run
try:
    baselines.lookup_rpm(clean_train, table_full.drop(("Reefer", "(0, 250]")))
    print("PROBLEM: missing table cell NOT caught")
except ValueError as e:
    print("missing table cell caught:", str(e)[:100])